In [7]:
import json
from kafka import KafkaConsumer

def run_full_consumer(config_file='config.json'):
    # Open and read the config file
    with open(config_file, 'r') as file:
        config = json.load(file)

    # Save the settings into variables
    kitchen_topic = config.get("kitchen_station_events", "kitchen_station_events")
    dispatch_topic = config.get("dispatch_events", "dispatch_events")
    servers = config.get("kafka_bootstrap_servers", "localhost:9092")

    # Use the variables to connect
    consumer = KafkaConsumer(
        kitchen_topic,
        dispatch_topic,
        bootstrap_servers=servers,
        value_deserializer=lambda v: json.loads(v.decode('utf-8')),
        auto_offset_reset='earliest'
    )

    print("Listening for all events... (Press Ctrl+C to stop)")

    try:
        for message in consumer:
            topic = message.topic
            data = message.value
            
            batch = data.get('batch_id')
            action = data.get('action')
            event_id = data.get('event_id')
            timestamp = data.get('event_timestamp')
            
            # Use the variables to check the topic
            if topic == kitchen_topic:
                station = data.get('station_id')
                recipe = data.get('recipe_id')
                weight = data.get('weight_kg')
                temp = data.get('temperature_celsius')
                ingredients = data.get('ingredients')
                details = f"EventID: {event_id} | Station: {station} | Recipe: {recipe} | Weight: {weight}kg | Temp: {temp}°C | Ingredients: {ingredients}"
            else:
                canteen = data.get('canteen_id')
                driver = data.get('driver_id')
                truck_temp = data.get('truck_temp_celsius')
                details = f"EventID: {event_id} | Canteen: {canteen} | Driver: {driver} | Truck Temp: {truck_temp}°C"
                
            print(f"[{topic}] {batch} -> {action} | {details} | Time: {timestamp}")
                
    except KeyboardInterrupt:
        consumer.close()

if __name__ == "__main__":
    run_full_consumer()

Listening for all events... (Press Ctrl+C to stop)
[kitchen_station_events] BATCH-1001 -> PREPARING | EventID: KIT-EVT-1001 | Station: prep_01 | Recipe: fish_curry | Weight: 13.52kg | Temp: 21.5°C | Ingredients: [{'item': 'white_fish', 'amount_kg': 6.76}, {'item': 'curry_paste', 'amount_kg': 4.06}, {'item': 'coconut_milk', 'amount_kg': 2.7}] | Time: 2026-03-15T06:01
[kitchen_station_events] BATCH-1002 -> PREPARING | EventID: KIT-EVT-1002 | Station: prep_02 | Recipe: chicken_rice | Weight: 38.34kg | Temp: 18.7°C | Ingredients: [{'item': 'chicken', 'amount_kg': 19.17}, {'item': 'rice', 'amount_kg': 15.34}, {'item': 'water', 'amount_kg': 3.83}] | Time: 2026-03-15T06:03
[kitchen_station_events] BATCH-1003 -> PREPARING | EventID: KIT-EVT-1003 | Station: prep_03 | Recipe: chicken_rice | Weight: 13.92kg | Temp: 16.5°C | Ingredients: [{'item': 'chicken', 'amount_kg': 8.45}, {'item': 'rice', 'amount_kg': 6.76}, {'item': 'water', 'amount_kg': 1.69}] | Time: 2026-03-15T06:08
[kitchen_station_even